In [ ]:
import torch
import torch.nn as nn
import bidirectional_dataset
import two_way_seq2seq
import trainer
import neptune
import itertools
import plotting

# Hyperparameter search
## Load the datasets

We train the model for 10 epochs on half the data from subjects 001, 004, 005, 007, 008, 009
and evaluate on 20% of the data for subjects 002 and 003

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_subjects = ["001", "004", "005", "007", "008", "009"]
validation_subjects = ["002", "003"]
training_proportion_to_use = 0.5
validation_proportion_to_use = 0.2

# in seconds
time_before_cutout = 1
cutout_duration = 1
time_after_cutout = 1

# defining the resampling stuff
original_freq = 200
resample_freq = 20

# create datasets
# creating training dataset
combined_train_dataset, individual_train_datasets, standardization_info = bidirectional_dataset.create_combined_dataset(subjects=training_subjects,
                                                                                proportion_to_use=training_proportion_to_use, time_before_cutout=time_before_cutout,
                                                                                cutout_duration=cutout_duration,
                                                                                resample_freq=resample_freq)

# creating validation dataset (standardized using the info from the training set)
combined_validation_dataset, individual_validation_datasets, _ = bidirectional_dataset.create_combined_dataset(subjects=validation_subjects,
                                                                                proportion_to_use=validation_proportion_to_use, time_before_cutout=time_before_cutout,
                                                                                cutout_duration=cutout_duration,
                                                                                resample_freq=resample_freq,
                                                                                standardization_info=standardization_info)


In [ ]:
n_channels = 7
cutout_duration_steps = cutout_duration * resample_freq
n_epoch = 10
epochs_with_teacher_forcing = 3

# HYPERPARAMETER GRID

batch_sizes = [512, 256]
learning_rates = [0.001, 0.01]
hidden_sizes = [1024, 512, 128]
encoder_dropout_probs = [0.2, 0.1, 0.0]
initial_teacher_forcing_probs = [0.2, 0.0]
num_layers = [4, 3, 2]


# perform the search (try all combinations of the above)
for batch_size, lr, hidden_size, encoder_dropout, initial_teacher_forcing, num_layers in \
    itertools.product(batch_sizes, learning_rates, hidden_sizes, encoder_dropout_probs, initial_teacher_forcing_probs, num_layers):
    # log run on neptune
    run_name = f"HYPERPARAMETER SEARCH - BS:{batch_size} HS:{hidden_size} LR:{lr} TF:{initial_teacher_forcing} D:{encoder_dropout} NL:{num_layers}"
    print(f"Starting run {run_name}")
    # create a new neptune run
    run = neptune.init_run(
        project="sleep-time-series/sleep-time-series",
        api_token="--omitted--",
        name=run_name,
        tags=["search linear candidate"]
    ) 

    # save params to neptune
    run["params"] = {
        "batch_size": batch_size,
        "hidden_size": hidden_size,
        "learning_rate": lr,
        "initial_teacher_forcing": initial_teacher_forcing,
        "encoder_droput": encoder_dropout,
        "n_epoch": n_epoch,
        "epochs_with_teacher_forcing": epochs_with_teacher_forcing,
        "num_layers": num_layers
    }

    run["training_subjects"] = ", ".join(training_subjects)
    run["validation_subjects"] = ", ".join(validation_subjects)
    run["training_proportion_used"] = training_proportion_to_use
    run["validation_proportion_used"] = validation_proportion_to_use

    # create new dataloaders (this is done because batch size changes)
    train_loader = torch.utils.data.DataLoader(combined_train_dataset, batch_size=batch_size, shuffle=True)
    validation_loader = torch.utils.data.DataLoader(combined_validation_dataset, batch_size=batch_size, shuffle=False)

    # create new model
    pred_model = two_way_seq2seq.TwoWaySeq2Seq(
        hidden_size, n_channels, cutout_duration_steps,
          encoder_dropout=encoder_dropout, num_layers=num_layers
          )
    # move model to parallel GPUs
    pred_model = nn.DataParallel(pred_model, device_ids = [0, 1])
    pred_model.to(device)
    optimizer = torch.optim.Adam(params = pred_model.parameters(), lr = lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = 3)
    loss_fn = nn.MSELoss(reduction="mean")

    # train
    trainer.pred_training_loop(pred_model, n_epoch, loss_fn, optimizer, train_loader, validation_loader, 
                                    neptune_run=run,
                                    device=device, scheduler=scheduler, 
                                    initial_teacher_forcing=initial_teacher_forcing, 
                                    epochs_with_teacher_forcing=epochs_with_teacher_forcing)
    
    # when training is done, create 3 plots of some predictions
    # to visually see the output of the model on neptune
    plot1 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 40, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    plot2 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 200, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    plot3 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 600, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
    
    run["plot1"].upload(plot1)
    run["plot2"].upload(plot2)
    run["plot3"].upload(plot3)
    run.stop()  